# Multi-Site Sales ETL Pipeline
Incremental ETL from three SQL Server store databases (SL, BO, PH) into a central PostgreSQL repository.
Outputs below are from the original run. Connection details come from environment variables (see `.env.example`).

In [3]:
#2
import os
import psycopg2
import pandas as pd

#make the connection
conn = psycopg2.connect(
    database=os.getenv("CENTRAL_DB"),
    user=os.getenv("CENTRAL_USER"),
    password=os.getenv("CENTRAL_PASSWORD"),
    host=os.getenv("CENTRAL_HOST"),
    port=os.getenv("CENTRAL_PORT", "5432")
)

# Creating a cursor object
cursor = conn.cursor()

#(check what siteIDs there are)
query = """
SELECT DISTINCT siteid
FROM site;
"""

df_sites = pd.read_sql(query, conn)

# Initialize empty list to store results
results = []

# Loop through each site ID
for site_id in df_sites['siteid']:
    # Query that finds max txid from SalesOrder table only
    # Joins Site -> SalesPerson -> SalesOrder -> Oline
    # Where siteid = x
    txid_query = f"""
    SELECT 
        s.siteid,
        MAX(so."TxID") AS max_txid
    FROM Site s
    INNER JOIN SalesPerson sp 
        ON s.siteid = sp.siteid
    INNER JOIN SalesOrder so 
        ON sp.spid = so.sellerid
    INNER JOIN OLine ol 
        ON so.orderid = ol.orderid
    WHERE s.siteid = '{site_id}'
    GROUP BY s.siteid;
    """
    
    # Execute query and append to results
    df_temp = pd.read_sql(txid_query, conn)
    results.append(df_temp)

# Combine all results into one dataframe
df_max_txid = pd.concat(results, ignore_index=True)

# Three print statements for contents of dataframe
print("Site-specific Maximum TxID Values:")
print(df_max_txid)

# Close connection
conn.close()


Site-specific Maximum TxID Values:
  siteid  max_txid
0     BO        25
1     SL        47
2     PH        28


In [4]:
#3
import os
import pymssql

# Database connection details
host = os.getenv("TX_HOST")
username = os.getenv("TX_USER")
password = os.getenv("TX_PASSWORD")

# Database mappings (database name -> siteid)
databases = {
    'pup_sl2': 'SL',
    'pup_bo2': 'BO',
    'pup_ph2': 'PH'
}

# Initialize empty list to store results
results = []

# Connect to each transactional database (loop through each connection)
for db_name, site_id in databases.items():
    # Get the max TxID for this site from df_max_txid
    max_txid_row = df_max_txid[df_max_txid['siteid'] == site_id]
    
    if not max_txid_row.empty:
        max_txid_value = max_txid_row['max_txid'].values[0]
        
        # Connect to the specific transactional database using pymssql
        conn = pymssql.connect(
            server=host,
            user=username,
            password=password,
            database=db_name
        )
        
        # SELECT orderid, order date, customer's name, sale total, siteid, salesperson's name
        # WHERE OrderID exceeds the TxID value from df_max_txid
        query = f"""
        SELECT 
            co.OrderID,
            co.ODate AS order_date,
            co.CustName AS customer_name,
            co.SaleTotal AS sale_total,
            co.SalesPerson AS salesperson_name
        FROM CustOrder co
        WHERE co.OrderID > {max_txid_value}
        ORDER BY co.OrderID;
        """
        
        # Execute query and append to results
        df_temp = pd.read_sql(query, conn)
        # Add siteid column since it's not in CustOrder table
        df_temp['siteid'] = site_id
        results.append(df_temp)
        
        # Close connection
        conn.close()

# Concat to single dataframe (called dfnewbs)
dfnewbs = pd.concat(results, ignore_index=True)

# Use Pandas sequential values for index column -> .reset_index()
dfnewbs = dfnewbs.reset_index(drop=True)

# Reorder columns to match expected output
dfnewbs = dfnewbs[['OrderID', 'order_date', 'customer_name', 'sale_total', 'siteid', 'salesperson_name']]

# Display the results
print("New Business Records (dfnewbs):")
print(dfnewbs)
print(f"\nTotal records: {len(dfnewbs)}")


New Business Records (dfnewbs):
    OrderID  order_date     customer_name  sale_total siteid salesperson_name
0        48  2022-09-04      Jaxon Wright       22.50     SL    Milburn Guido
1        49  2022-09-04      Marla Gibson    17995.00     SL    Milburn Guido
2        50  2022-09-04    Trucker Vogvog    18074.00     SL    Milburn Guido
3        54  2022-09-04     Ace Henderson    19203.85     SL    Alberto Dimas
4        55  2022-09-05  Mugwort Evermore    26504.80     SL    Milburn Guido
5        26  2022-09-03         Bog McGee    25939.00     BO  Roksana Antonia
6        27  2022-09-04    Brittany Adams      158.00     BO     Kelan Fowler
7        29  2022-09-04     Edwin Adamson    25990.00     PH       Robby Lexy
8        31  2022-09-04      Eldon Lowery    18871.80     PH  Miah Wittington
9        32  2022-09-04    Shelby Ramirez    18444.00     PH       Robby Lexy
10       33  2022-09-04   Carter Franklin       79.00     PH       Robby Lexy

Total records: 11


In [5]:
#4

import os
# Split customer_name from dfnewbs into first and last names
dfnewbs[['first_name', 'last_name']] = dfnewbs['customer_name'].str.split(' ', n=1, expand=True)

# Make df_central_names by querying first and last names from customer table in central repository
central_conn = psycopg2.connect(
    database=os.getenv("CENTRAL_DB"),
    user=os.getenv("CENTRAL_USER"),
    password=os.getenv("CENTRAL_PASSWORD"),
    host=os.getenv("CENTRAL_HOST"),
    port=os.getenv("CENTRAL_PORT", "5432")
)

query_central = """
SELECT custfname, custlname
FROM Customer;
"""

df_central_names = pd.read_sql(query_central, central_conn)
central_conn.close()

# Go through each customer and check if they are in the central repository
for index, row in dfnewbs.iterrows():
    first = row['first_name']
    last = row['last_name']
    
    # Check if customer exists in central repository
    is_in_central = ((df_central_names['custfname'] == first) & 
                     (df_central_names['custlname'] == last)).any()
    
    if is_in_central:
        print(f"{first} {last} is in the central repository")
    else:
        print(f"{first} {last} is not in the central repository")


Jaxon Wright is in the central repository
Marla Gibson is not in the central repository
Trucker Vogvog is not in the central repository
Ace Henderson is in the central repository
Mugwort Evermore is not in the central repository
Bog McGee is not in the central repository
Brittany Adams is in the central repository
Edwin Adamson is not in the central repository
Eldon Lowery is in the central repository
Shelby Ramirez is not in the central repository
Carter Franklin is in the central repository


## Step 5 — Load
Matches customers on name **and** phone or email, updates returning customers, inserts new ones, then loads orders and order lines.

**Timestamp fix:** an earlier run stamped rows with Python's `datetime.now()` (local Mountain time) while PostgreSQL runs in UTC, so evening loads landed on the server's "yesterday" and the verification queries returned nothing. This version takes `CURRENT_TIMESTAMP` from the database so loading and querying share one clock.

In [23]:
#5

import os
import psycopg2
import pymssql
import pandas as pd
import numpy as np
from psycopg2.extensions import register_adapter, AsIs

# Register NumPy type adapters for psycopg2 compatibility
register_adapter(np.int64, lambda val: AsIs(val))
register_adapter(np.int32, lambda val: AsIs(val))
register_adapter(np.float64, lambda val: AsIs(val))
register_adapter(np.float32, lambda val: AsIs(val))

# --- Connection Details ---
pg_host = os.getenv("CENTRAL_HOST")
pg_db = os.getenv("CENTRAL_DB")
pg_user = os.getenv("CENTRAL_USER")
pg_pass = os.getenv("CENTRAL_PASSWORD")
pg_port = os.getenv("CENTRAL_PORT", "5432")

# Transactional databases (MS SQL Server)
sql_server_host = os.getenv("TX_HOST")
sql_server_user = os.getenv("TX_USER")
sql_server_pass = os.getenv("TX_PASSWORD")

# List of transactional databases with their site IDs
transactional_databases = [
    ("pup_sl2", "SL"),
    ("pup_bo2", "BO"),
    ("pup_ph2", "PH")
]

# --- Main ETL Process ---
pg_conn = None
try:
    # Connect to central repository
    pg_conn = psycopg2.connect(host=pg_host, database=pg_db, user=pg_user, password=pg_pass, port=pg_port)
    pg_cursor = pg_conn.cursor()
    print("Successfully connected to the central repository (PostgreSQL).\n")

    # Get current timestamp from the DATABASE
    pg_cursor.execute("SELECT CURRENT_TIMESTAMP;")
    current_timestamp = pg_cursor.fetchone()[0]
    print(f"Database timestamp: {current_timestamp}\n")

    # Step 1: Get max TxID for each site from central repository
    max_txid_query = """
    SELECT s.siteid, COALESCE(MAX(so."TxID"), 0) as max_txid 
    FROM Site s
    INNER JOIN SalesPerson sp 
        ON s.siteid = sp.siteid
    LEFT JOIN SalesOrder so 
        ON sp.spid = so.sellerid
    GROUP BY s.siteid
    """
    pg_cursor.execute(max_txid_query)
    max_txid_results = pg_cursor.fetchall()
    max_txid_dict = {row[0]: row[1] for row in max_txid_results}
    print("Maximum TxID values by site:")
    for site, max_val in max_txid_dict.items():
        print(f"  {site}: {max_val}")
    
    # Step 2: Extract new orders and order lines from each transactional database
    df_orders_list = []
    df_orderlines_all = pd.DataFrame()
    
    for db_name, site_id in transactional_databases:
        tx_conn = None
        try:
            print(f"\n--- Extracting new data from {db_name} ({site_id}) ---")
            tx_conn = pymssql.connect(server=sql_server_host, user=sql_server_user, 
                                     password=sql_server_pass, database=db_name)
            
            # Get the maximum TxID already ingested for this site
            max_txid = max_txid_dict.get(site_id, 0)
            
            # Query new orders (lowercase column names)
            query_orders = f"SELECT * FROM CustOrder WHERE orderid > {max_txid}"
            df_orders = pd.read_sql(query_orders, tx_conn)
            df_orders['siteid'] = site_id
            df_orders_list.append(df_orders)
            print(f"  Found {len(df_orders)} new orders")
            
            # Query new order lines (lowercase column names)
            query_orderlines = f"SELECT * FROM Order_Line WHERE orderid > {max_txid}"
            df_orderlines_temp = pd.read_sql(query_orderlines, tx_conn)
            df_orderlines_temp['siteid'] = site_id
            df_orderlines_all = pd.concat([df_orderlines_all, df_orderlines_temp], ignore_index=True)
            print(f"  Found {len(df_orderlines_temp)} new order line items")
            
        except pymssql.Error as error:
            print(f"Error connecting to {db_name}: {error}")
        finally:
            if tx_conn:
                tx_conn.close()
    
    # Step 3: Concatenate all orders into single dataframe
    dfnewbs = pd.concat(df_orders_list, ignore_index=True)
    print(f"\n--- Total new orders to process: {len(dfnewbs)} ---\n")
    
    # Step 4: Get ALL customer info from central database
    customer_query = """
    SELECT custid, custfname, custlname, custphone, custemail, custstate, contactpref, pmtid 
    FROM customer
    """
    df_contact_central = pd.read_sql(customer_query, pg_conn)
    
    # Step 5: Initialize lists for customers to update vs. add
    df_update = []
    df_add = []
    customer_mapping = {}  # Maps order index to custid
    
    # Step 6: Process each order to determine if customer exists
    for idx, order in dfnewbs.iterrows():
        # Split customer name into first and last (lowercase column name)
        full_name = str(order['custname']).split()
        first_name = " ".join(full_name[:-1]) if len(full_name) > 1 else full_name[0]
        last_name = full_name[-1] if len(full_name) > 1 else ''
        
        # Check if customer exists in central database
        name_matches = df_contact_central[
            (df_contact_central['custfname'] == first_name) & 
            (df_contact_central['custlname'] == last_name)
        ]
        
        customer_exists = False
        if not name_matches.empty:
            # Check if phone OR email also matches (lowercase column names)
            contact_matches = name_matches[
                (name_matches['custphone'] == order.get('custphone')) | 
                (name_matches['custemail'] == order.get('custemail', ''))
            ]
            
            if not contact_matches.empty:
                # Customer exists - add to update list
                customer_exists = True
                custid = contact_matches.iloc[0]['custid']
                customer_mapping[idx] = custid
                
                df_update.append({
                    'custid': custid,
                    'custphone': order.get('custphone'),
                    'custemail': order.get('custemail', ''),
                    'custstate': order.get('custstate', ''),
                    'contactpref': order.get('contactpref', ''),
                    'pmtid': order.get('pmtid')
                })
        
        if not customer_exists:
            # New customer - add to insert list (lowercase column names)
            df_add.append({
                'idx': idx,  # Track which order this belongs to
                'custfname': first_name,
                'custlname': last_name,
                'custphone': order.get('custphone'),
                'custemail': order.get('custemail', ''),
                'custstate': order.get('custstate', ''),
                'contactpref': order.get('contactpref', ''),
                'pmtid': order.get('pmtid')
            })
    
    # Step 7: Update existing customers
    for customer in df_update:
        update_sql = """
        UPDATE customer 
        SET custphone = %s, custemail = %s, custstate = %s, contactpref = %s, 
            pmtid = %s, updatedstamp = %s
        WHERE custid = %s
        """
        pg_cursor.execute(update_sql, (
            customer['custphone'], customer['custemail'], customer['custstate'],
            customer['contactpref'], customer['pmtid'], current_timestamp, customer['custid']
        ))
        print(f"Customer ID {customer['custid']} updated")
    
    # Step 8: Insert new customers
    for customer in df_add:
        insert_sql = """
        INSERT INTO customer (custfname, custlname, custphone, custemail, custstate, 
                             contactpref, pmtid, addedstamp, updatedstamp)
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
        RETURNING custid
        """
        pg_cursor.execute(insert_sql, (
            customer['custfname'], customer['custlname'], customer['custphone'],
            customer['custemail'], customer['custstate'], customer['contactpref'],
            customer['pmtid'], current_timestamp, current_timestamp
        ))
        new_custid = pg_cursor.fetchone()[0]
        customer_mapping[customer['idx']] = new_custid
        print(f"Customer {customer['custfname']} {customer['custlname']} added")
    
    pg_conn.commit()
    
    # Step 9: Process sales orders
    for idx, order in dfnewbs.iterrows():
        try:
            # Get customer ID
            custid = customer_mapping.get(idx)
            if not custid:
                print(f"Warning: Could not find customer for order {order['orderid']}")
                continue
            
            # Get site ID
            siteid = order['siteid']
            
            # Split salesperson name (lowercase column name)
            sp_full_name = str(order['salesperson']).split()
            sp_fname = " ".join(sp_full_name[:-1]) if len(sp_full_name) > 1 else sp_full_name[0]
            sp_lname = sp_full_name[-1] if len(sp_full_name) > 1 else ''
            
            # Find salesperson ID
            sp_query = """
            SELECT spid FROM salesperson 
            WHERE spfname = %s AND splname = %s AND siteid = %s
            """
            pg_cursor.execute(sp_query, (sp_fname, sp_lname, siteid))
            sp_result = pg_cursor.fetchone()
            
            if not sp_result:
                print(f"Warning: Salesperson '{order['salesperson']}' not found for site {siteid}")
                continue
            
            spid = sp_result[0]
            
            # Insert sales order
            insert_order_sql = """
            INSERT INTO salesorder (custid, sellerid, odate, ototal, ordsatisfac, delsatisfac, 
                                   "TxID", "TxIngestTimestamp")
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
            RETURNING orderid
            """
            pg_cursor.execute(insert_order_sql, (
                custid, spid, order['odate'], order.get('saletotal'), None, None,
                order['orderid'], current_timestamp
            ))
            new_orderid = pg_cursor.fetchone()[0]
            
            # Process order line items (lowercase column names)
            order_lines = df_orderlines_all[
                (df_orderlines_all['orderid'] == order['orderid']) & 
                (df_orderlines_all['siteid'] == siteid)
            ]
            
            for _, line in order_lines.iterrows():
                # Get product ID (lowercase column name)
                prod_query = "SELECT prodid FROM product WHERE prodname = %s"
                pg_cursor.execute(prod_query, (line['prodname'],))
                prod_result = pg_cursor.fetchone()
                
                if not prod_result:
                    print(f"Warning: Product '{line['prodname']}' not found")
                    continue
                
                prodid = prod_result[0]
                
                # Calculate line price and total (lowercase column names)
                linetotal = line.get('linetotal', 0)
                lineprice = linetotal / line['qty'] if line['qty'] > 0 else 0
                
                # Insert order line
                insert_line_sql = """
                INSERT INTO oline (orderid, productid, lineqty, lineprice, linetotal, 
                                  builtdate, promisedate, deldate)
                VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
                """
                pg_cursor.execute(insert_line_sql, (
                    new_orderid, prodid, line['qty'], lineprice, linetotal,
                    None, None, None
                ))
            
            pg_conn.commit()
            print(f"Order {order['orderid']} from {siteid} successfully recorded as OrderID {new_orderid}")
            
        except Exception as e:
            print(f"Error processing order {order['orderid']}: {e}")
            pg_conn.rollback()

except (Exception, psycopg2.DatabaseError) as error:
    print(f"\nCritical error: {error}")
    if pg_conn:
        pg_conn.rollback()
finally:
    if pg_conn:
        if 'pg_cursor' in locals() and pg_cursor:
            pg_cursor.close()
        pg_conn.close()
        print("\nCentral repository connection closed.")


Successfully connected to the central repository (PostgreSQL).

Database timestamp: 2025-11-15 05:57:28.301573+00:00

Maximum TxID values by site:
  BO: 25
  SL: 47
  PH: 28

--- Extracting new data from pup_sl2 (SL) ---


  Found 5 new orders
  Found 12 new order line items

--- Extracting new data from pup_bo2 (BO) ---


  Found 2 new orders
  Found 4 new order line items

--- Extracting new data from pup_ph2 (PH) ---


  Found 4 new orders
  Found 9 new order line items

--- Total new orders to process: 11 ---

Customer ID 4733 updated
Customer ID 4736 updated
Customer ID 5221 updated
Customer ID 4759 updated
Customer ID 5201 updated
Customer Marla Gibson added
Customer Trucker Vogvog added
Customer Mugwort Evermore added
Customer Bog McGee added
Customer Edwin Adamson added
Customer Shelby Ramirez added
Order 48 from SL successfully recorded as OrderID 31109
Order 49 from SL successfully recorded as OrderID 31110
Order 50 from SL successfully recorded as OrderID 31111
Order 54 from SL successfully recorded as OrderID 31112
Order 55 from SL successfully recorded as OrderID 31113
Order 26 from BO successfully recorded as OrderID 31114
Order 27 from BO successfully recorded as OrderID 31115
Order 29 from PH successfully recorded as OrderID 31116
Order 31 from PH successfully recorded as OrderID 31117
Order 32 from PH successfully recorded as OrderID 31118
Order 33 from PH successfully recorded as Order

In [24]:
#6

import os
# Connect to central repository
central_conn = psycopg2.connect(
    database=os.getenv("CENTRAL_DB"),
    user=os.getenv("CENTRAL_USER"),
    password=os.getenv("CENTRAL_PASSWORD"),
    host=os.getenv("CENTRAL_HOST"),
    port=os.getenv("CENTRAL_PORT", "5432")
)

# Query for customers added or updated today
query_step6 = """
SELECT 
    so.orderid,
    c.custfname,
    c.custlname,
    c.custstate,
    c.custemail,
    c.custphone,
    c.pmtid,
    c.contactpref
FROM customer c
INNER JOIN salesorder so ON c.custid = so.custid
WHERE DATE(c.addedstamp) = CURRENT_DATE 
   OR DATE(c.updatedstamp) = CURRENT_DATE
ORDER BY so.orderid;
"""

# Read query into dataframe with orderid as index
df_step6 = pd.read_sql(query_step6, central_conn, index_col='orderid')

# Display results
print("Customers Added or Updated Today:")
print(df_step6)

# Close connection
central_conn.close()


Customers Added or Updated Today:
        custfname  custlname custstate                     custemail  \
orderid                                                                
29771       Jaxon     Wright        UT          jwright215@yahoo.com   
29774         Ace  Henderson        UT      acehenderson63@yahoo.com   
29795       Eldon     Lowery        AZ             elowery@gmail.com   
30202      Carter   Franklin        CA  carterfranklin68@outlook.com   
30220    Brittany      Adams        ID           badams501@gmail.com   
31109       Jaxon     Wright        UT          jwright215@yahoo.com   
31110       Marla     Gibson        UT            marlag@hotmail.com   
31111     Trucker     Vogvog        WA  truckervogvog8441@google.com   
31112         Ace  Henderson        UT      acehenderson63@yahoo.com   
31113     Mugwort   Evermore        UT           mugever88@gmail.com   
31114         Bog      McGee        ID             bogger17@gmail.tv   
31115    Brittany      Adams  

In [25]:
#7
import os
import psycopg2
import pandas as pd

# Connect to central repository
central_conn = psycopg2.connect(
    database=os.getenv("CENTRAL_DB"),
    user=os.getenv("CENTRAL_USER"),
    password=os.getenv("CENTRAL_PASSWORD"),
    host=os.getenv("CENTRAL_HOST"),
    port=os.getenv("CENTRAL_PORT", "5432")
)

# Query for site sales summary for orders ingested today
# Need to join through salesperson to get to site
query_step7 = """
SELECT 
    si.siteid,
    si.sitename,
    SUM(so.ototal) AS total_revenue,
    COUNT(so.orderid) AS total_sales
FROM site si
INNER JOIN salesperson sp ON si.siteid = sp.siteid
INNER JOIN salesorder so ON sp.spid = so.sellerid
WHERE so."TxIngestTimestamp"::date = CURRENT_DATE
GROUP BY si.siteid, si.sitename
ORDER BY total_revenue DESC;
"""

# Read query into dataframe with siteid as index
df_step7 = pd.read_sql(query_step7, central_conn, index_col='siteid')

# Display results
print("Site Sales Summary for Orders Ingested Today:")
print(df_step7)

# Close connection
central_conn.close()


Site Sales Summary for Orders Ingested Today:
              sitename  total_revenue  total_sales
siteid                                            
SL      Salt Lake City       81800.15            5
PH             Phoenix       63384.80            4
BO               Boise       26097.00            2
